# ColabUMA Conf — UMAによる配座探索

Metaの機械学習ポテンシャル **UMA** で、金属錯体と基質の会合体などの **配座探索** を行い、エネルギーの低い順に数個（既定は5個）の構造を出すノートブックです。UMA の学習データ（OMol25）は金属錯体を多く含むので、力場や半経験的手法が苦手な金属まわりもそのまま扱えます。コードを書く必要はありません。

1. 入力構造を最適化し、**結合のつながり**（どの原子どうしが結合しているか）を読み取る
2. **高温の分子動力学（MD）** を何本か走らせ、途中の構造（スナップショット）を集める
3. スナップショットを粗く最適化し、エネルギーの低いものだけを最後まで最適化する
4. 結合のつながりが変わった構造と、同じ配座の重複を除き、エネルギーの低い順に並べる

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 初回のみ、左サイドバーの🔑（シークレット）に名前 `HF_TOKEN`、値にHugging Faceのアクセストークンを登録し、「ノートブックからのアクセス」をオンにする
3. 下の **1. Input form** で出力名、電荷、スピン多重度、探索の条件を指定する
4. **ランタイム → すべて実行** を押す
5. **2. File selection and checks** にファイル選択ボタンが出るので、**金属錯体に基質を置いた構造** を xyz または Gaussian の gjf / com 1つで選ぶ。あとは放置できます
6. 下のセルで配座の表・3D表示・エネルギーの図を確認し、最後のセルで結果のZIPをダウンロードする

gjfからは構造だけを読み込みます。電荷と多重度はフォームの値（錯体と基質を合わせた全体の値）を使います。2回目以降は手順3〜6を繰り返すだけです（インストールとモデル読み込みは自動でスキップされます）。ファイル選択で Cancel upload を押すと、前回のファイルをそのまま使います。

> **事前準備**：モデルの重み [facebook/UMA](https://huggingface.co/facebook/UMA) はアクセス申請が必要です。Hugging Faceにログインしてページ上でFAIR Chemistry License v1に同意し、承認後に [Access Tokens](https://huggingface.co/settings/tokens) で Read 権限のトークンを作成してください。

## 基質の付き方は指定しません

基質が金属に配位しているか、水素結合やπ相互作用で付いているだけかは、指定する必要がありません。最適化した入力構造で結合している原子の組（金属と配位原子の組も含む）をそのまま保ちます。

- 入力で基質が **金属に配位している** → 配位を保ったまま、配座だけが変わる
- 入力で基質が **付いているだけ**（結合していない）→ 基質の位置と向きも変わる。離れていかないように、分子全体を球の壁で囲む

MD のあいだは、結合が伸びすぎたときだけ引き戻す拘束を掛けます。最適化のあと、結合が切れた・新しくできた・立体中心が反転した構造は捨てます。`keep_coordination` をオフにすると、金属が関わる組は見張らなくなり、配位子や基質が金属に付いたり離れたりした構造も残ります。

## 探索の条件

| 項目 | 既定値 | 説明 |
|---|---|---|
| `n_conformers` | 5 | 出力する配座の数 |
| `md_temperature` | 600 | MD の温度（K）。高いほど大きく動くが、壊れた構造が増える |
| `md_time_ps` | 20 | MD の時間（ps、全部の MD の合計）。刻みは 2 fs（MD のあいだだけ水素の質量を 4 にする） |
| `n_replicas` | 4 | MD の本数。初速を変えて同じ構造から始める |
| `n_snapshots` | 100 | MD から等間隔に取り出す構造の数 |
| `n_refine` | 20 | 粗い最適化のあと、最後まで最適化する候補の数 |
| `keep_coordination` | オン | 入力構造の金属まわりの結合を保つ |
| 収束判定 | normal (0.02) | 最後の最適化の fmax（eV/Å） |

計算時間は、力の計算の回数でほぼ決まります。既定値では MD が1万回、最適化が合わせて数千〜1万回程度です。進み具合のバーに残り時間の目安が出ます。時間がかかりすぎるときは `md_time_ps`、`n_snapshots`、`n_refine` を減らしてください。

## 出力（ZIPの中身）

| ファイル | 内容 |
|---|---|
| `<出力名>_conformers.xyz` | 出力する配座（エネルギーの低い順、複数構造xyz）。1番目に重ね合わせてある |
| `<出力名>_conf1.xyz` … | 各配座を1つずつ |
| `<出力名>_conformers.csv` | ΔE（kcal/mol）、298 K での存在比、エネルギー、1番目との RMSD、収束 |
| `<出力名>_ensemble.xyz` | 最後まで最適化して残った配座すべて（出力する数より後のものも含む） |
| `<出力名>_input_opt.xyz` | 入力構造を最適化したもの |
| `<出力名>_dft.csv` | DFT 一点計算のエネルギー（DFT 一点計算オンのとき） |
| `<出力名>_log.txt` | 入力条件、ソフトのバージョン、探索の経過、各配座のエネルギー |

## 注意事項

- 高温 MD による探索は、すべての配座を見つける保証がありません。大きく柔らかい分子では、`md_time_ps` を長くしたり `random_seed` を変えたりして、同じ最安定構造が出るか確かめてください。
- 得られるのは **入力構造と同じ結合のつながりを持つ** 配座です。配位の仕方が違う異性体（別の配位サイト、cis / trans など）は、それぞれを入力にして別々に探索し、エネルギーを比べてください。
- 存在比は、出力した配座の中での ΔE（電子エネルギー）だけから求めた目安です。振動や溶媒の寄与は入っていません。
- 電荷とスピン多重度は全構造で固定です。スピン状態が違う可能性があるときは、多重度を変えて別々に実行してください。
- UMAは機械学習ポテンシャルであり、DFTの代わりにはなりません。最終的なエネルギー比較はDFTで確認してください（`run_dft` で出力した配座の一点計算ができます）。
- 重要な構造は [ColabUMA_Opt](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Opt.ipynb) で振動計算をして、極小であることを確かめてください。
- 周期系（結晶・表面）には対応していません。

## ライセンス

- このノートと共通コード（`colabuma`）：[MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- UMAのモデル重み：[FAIR Chemistry License v1](https://huggingface.co/facebook/UMA) に従います
- 実行時にインストールするライブラリは、それぞれのライセンスに従います。一覧は [README のライセンス](https://github.com/RChibaChiba/ColabChem/blob/main/README.ja.md#ライセンス) にあります

## 引用

論文で使う場合は、以下を引用してください。

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

DFT の一点計算を使った場合は、次も引用してください。

- PySCF（Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020)）
- GPU4PySCF（X. Wu *et al.*, arXiv:2404.09452 (2024)）

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick an xyz or gjf file in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
#@markdown Charge and spin multiplicity of the whole system (metal complex and substrate together).
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### Conformer search
#@markdown Number of conformers to report (lowest energy first).
n_conformers = 5  #@param {type:"integer"}
#@markdown Temperature (K) and total time (ps, all replicas together) of the MD that explores the conformers. Hotter and longer finds more, but takes longer.
md_temperature = 600  #@param {type:"number"}
md_time_ps = 20  #@param {type:"number"}
#@markdown Number of independent MD runs (the total time is split between them) and of snapshots taken from them.
n_replicas = 4  #@param {type:"integer"}
n_snapshots = 100  #@param {type:"integer"}
#@markdown After a quick optimization of all snapshots, this many of the lowest are optimized fully.
n_refine = 20  #@param {type:"integer"}
#@markdown Keep the metal–ligand contacts of the input structure. Turn off to let ligands or the substrate bind to and leave the metal.
keep_coordination = True  #@param {type:"boolean"}
random_seed = 0  #@param {type:"integer"}

#@markdown ### Settings
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown Convergence threshold fmax (eV/Å) of the full optimization. normal is close to Gaussian's default, tight to Gaussian's opt=tight.
convergence = "normal (0.02)"  #@param ["loose (0.05)", "normal (0.02)", "tight (0.001)", "custom"]
#@markdown fmax (eV/Å) used when custom is chosen.
fmax_custom = 0.02  #@param {type:"number"}
#@markdown The maximum number of steps is per structure.
max_steps = 500  #@param {type:"integer"}

#@markdown ### DFT single point (gpu4pyscf)
#@markdown Computes the DFT energies of the reported conformers. ωB97M-V/def2-TZVPD is the level of theory of UMA's training data (OMol25). def2-TZVPD can take tens of minutes per point on a T4.
run_dft = False  #@param {type:"boolean"}
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V (same as OMol25)", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]
#@markdown Time limit for DFT (minutes, all structures together). Past it, DFT is skipped and the UMA results are saved. 0 means no limit (DFT still stops before the Colab runtime is likely to end).
dft_time_limit = 60  #@param {type:"integer"}

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection and the token check happen here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import conf, dft, env, opt, output, structure, ui
from colabuma.ui import ColabUMAError, ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
st = conf.parse_settings(n_conformers, md_temperature, md_time_ps, n_replicas, n_snapshots, n_refine,
                         keep_coordination, random_seed)
if run_dft:
    dft.parse_method(dft_functional, dft_basis)
    if dft_time_limit < 0:
        raise ColabUMAError("The DFT time limit must be 0 or more.", "0 means no limit.",
                            ja="DFT の時間の上限は0以上にしてください。", ja_hint="0 なら上限なしです。")
env.check_gpu()
env.ensure_token()
upload_name, upload_data = structure.upload()
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install and login) { display-mode: "form" }
DEVICE = env.setup(["dft"] if run_dft else [])

In [ ]:
#@title 4. Load the model { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. Prepare the structures { display-mode: "form" }
atoms0, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity)
ok(f"Initial structure ready: {atoms0.get_chemical_formula()} ({len(atoms0)} atoms).")
structure.show_numbered(atoms0)
if run_dft:
    dft.check_memory([("", atoms0)], charge, multiplicity, dft.parse_method(dft_functional, dft_basis)[1])

In [ ]:
#@title 6. Conformer search { display-mode: "form" }
#@markdown Optimizes the input, runs the high-temperature MD, then optimizes the snapshots in two rounds. The bars show the time left.
conf_res = conf.run(atoms0, calc, job, st)

In [ ]:
#@title 7. Results { display-mode: "form" }
files, log = conf.report(conf_res, job)

In [ ]:
#@title 8. DFT single points { display-mode: "form" }
dft_log = []
if run_dft:
    _points, _uma_rows = conf.dft_points(conf_res)
    _f, dft_log = dft.run_optional(_points, charge, multiplicity, dft_functional, dft_basis, _uma_rows, dft_time_limit)
    files.update(_f)
else:
    print("DFT single points are off.")

In [ ]:
#@title 9. Save { display-mode: "form" }
files["log.txt"] = output.log_text("ColabUMA Conf log", source_desc, job, [log, dft_log])
zip_path = output.save_zip(job.out_prefix, files)